In [43]:
# WRAP CODE CSS

# Source - https://stackoverflow.com/a
# Posted by Bon Ryu, modified by community. See post 'Timeline' for change history
# Retrieved 2025-12-29, License - CC BY-SA 4.0

from IPython.display import HTML, display

def set_css():
  display(HTML('''
  <style>
    pre {
        white-space: pre-wrap;
    }
  </style>
  '''))
get_ipython().events.register('pre_run_cell', set_css)

In [44]:
# READ EVENTS CSV

from datetime import datetime

def read_events_csv(path):
    events = {}

    with open(path, newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            events[row["Event_ID"]] = {
                "name": row["Event_name"],
                "type": row["Type"],
                "start": row["Start_date"],
                "end": row["End_date"]
            }

    return events


events = read_events_csv("/content/drive/MyDrive/events.csv")
# events["covid-19"]["start"]

In [45]:
#READ TARGET WORDS CSV

def read_target_words_csv(path):
    pairs = []
    with open(path, newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            pairs.append((row["Word"], row["Event_ID"]))
    return pairs

pairs = read_target_words_csv("/content/drive/MyDrive/target_words.csv")
#find event directly from word
word_to_event = dict(pairs)

In [46]:
import csv

events = read_events_csv("/content/drive/MyDrive/events.csv")
target_pairs = read_target_words_csv(
    "/content/drive/MyDrive/target_words.csv"
)

In [47]:
!pip install praw
import praw
import csv
from datetime import datetime
from google.colab import drive
import json

In [48]:
#secret key

# Source - https://stackoverflow.com/a
# Posted by Bob Smith
# Retrieved 2025-12-29, License - CC BY-SA 4.0

from getpass import getpass
reddit_secret_key = getpass('Enter the secret reddit key value: ')
reddit_client_id = getpass('Enter the secret reddit client id: ')

Enter the secret reddit key value: ··········
Enter the secret reddit client id: ··········


In [49]:
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [50]:
!pip install langdetect
from langdetect import detect, DetectorFactory
DetectorFactory.seed = 0

In [51]:
#CONVERT JSON TO CSV

def json_to_csv(json_path, csv_path, word, event_id, events_dict):
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    headers = ["word", "event", "register", "period", "sentence", "timestamp"]

    with open(csv_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(headers)

        for item in data:
            sentence = item.get("selftext", "")
            if not sentence:
                continue

            ts = item["created_utc"]

            timestamp = datetime.utcfromtimestamp(ts).strftime(
                "%Y-%m-%dT%H:%M:%SZ"
            )

            period = assign_period(ts, events_dict[event_id])

            writer.writerow([
                word,
                event_id,
                "informal",
                period,
                sentence,
                timestamp
            ])


In [52]:
#REDDIT DATASET CONVERT UTC TO DESIRED FORMAT

def extract_year(date_str):
    if not date_str:
        return None
    return int("20" + date_str.split(".")[-1])


In [53]:
# period column in csv

def assign_period(timestamp_utc, event_info):
    year = datetime.utcfromtimestamp(timestamp_utc).year

    start_year = extract_year(event_info["start"])
    end_year = extract_year(event_info["end"])

    if start_year and year < start_year:
        return "before"

    if end_year and year > end_year:
        return "after"

    return "during"

In [54]:
import os

def get_search(query_name, event_name, maxPosts, drive_path="/content/drive/MyDrive/ColabNotebooks/ColabOutputs"):

  reddit = praw.Reddit(
      client_id=reddit_client_id,
      client_secret=reddit_secret_key,
      user_agent="keyword_searcher by u/DuckApprehensive203",
  )

  output_path = os.path.join(drive_path, f"reddit_{query_name}_{maxPosts}.json")

  query = query_name
  max_posts = maxPosts

  posts = []

  for submission in reddit.subreddit("all").search(query, sort="new", limit=max_posts):
      text_to_check = (submission.title or "") + " " + (submission.selftext or "")
      try:
          lang = detect(text_to_check)
      except:
          lang = "unknown"

      if lang == "en":
          post_data = {
              "id": submission.id,
              "title": submission.title,
              "subreddit": submission.subreddit.display_name,
              "score": submission.score,
              "url": submission.url,
              "created_utc": submission.created_utc,
              "author": str(submission.author),
              "num_comments": submission.num_comments,
              "selftext": submission.selftext,
          }
          posts.append(post_data)

  print(json.dumps(posts, indent=2))

  with open(output_path, "w", encoding="utf-8") as f:
    json.dump(posts, f, indent=2, ensure_ascii=False)

  csv_output_path = output_path.replace(".json", ".csv")
  json_to_csv(output_path, csv_output_path, query_name, event_name, events)


get_search('algorithm', word_to_event['algorithm'], 10000)
get_search('binge', word_to_event['binge'], 10000)
get_search('bubble', word_to_event['bubble'], 10000)
get_search('drop', word_to_event['drop'], 10000)
get_search('model', word_to_event['model'], 10000)
get_search('prompt', word_to_event['prompt'], 10000)
get_search('remote', word_to_event['remote'], 10000)
get_search('spoiler', word_to_event['spoiler'], 10000)
get_search('stream', word_to_event['stream'], 10000)
get_search('tag', word_to_event['tag'], 10000)
get_search('token', word_to_event['token'], 10000)
get_search('tweet', word_to_event['tweet'], 10000)
get_search('variant', word_to_event['variant'], 10000)

It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.

It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.

It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.

/tmp/ipython-input-1164839570.py:20: DeprecationWarning: datetime.datetime.utcfromtimestamp() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.fromtimestamp(timestamp, datetime.UTC).
  timestamp = datetime.utcfromtimestamp(ts).strftime(
/tmp/ipython

[
  {
    "id": "1pywn0h",
    "title": "MIT Battlecode (programming competition) starts in 1 week!",
    "subreddit": "mit",
    "score": 1,
    "url": "https://www.reddit.com/r/mit/comments/1pywn0h/mit_battlecode_programming_competition_starts_in/",
    "created_utc": 1767038829.0,
    "author": "battlecode-devs",
    "num_comments": 0,
    "selftext": "Battlecode is a real-time strategy game where you\u2019ll use game theory, pathfinding, and distributed algorithms to build an autonomous team of robots that will have to defeat an opposing team.\n\nAnyone is welcome to compete in teams of 1-4, for a share of the **$20k** prize pool, and the top team will get a **guaranteed internship** with our Platinum Sponsor,\u00a0[Amplitude](https://amplitude.com/)! The top 16 student teams will also be flown to MIT for the Final Tournament on 1/31 for free.\n\nNo experience is needed beyond basic programming skills! Bots are written in Java and/or Python, though we recommend Java. We\u2019ll wal

It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.

It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.

It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.



[
  {
    "id": "1pywdc6",
    "title": "First binge in 7 months",
    "subreddit": "BingeEatingDisorder",
    "score": 2,
    "url": "https://www.reddit.com/r/BingeEatingDisorder/comments/1pywdc6/first_binge_in_7_months/",
    "created_utc": 1767038233.0,
    "author": "Puzzleheaded_Tap6344",
    "num_comments": 1,
    "selftext": "I went to bed too hungry last night. Tossed and turned. Head was aching. Told myself to wait to eat in the morning.. kitchen was closed. Surely I ate enough for dinner. \nWoke up, kissed my husband goodbye for work- then I went to the cupboard. The fridge. The garbage. Keep in mind this was after a \u201chealthy\u201d breakfast I had planned. Ended the breakfast feeling more famished than before. 10000 calories later, I am in more pain than I ever remember. I just came back from a walk and the blood flow further hardened my stomach and the pain. Why do I forget the pain and suffering so quickly. I had so much progress and got cocky with my ability to eat le

It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.

It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.

It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.



...

[
  {
    "id": "1pywg3g",
    "title": "Indian Phosphate Limited Sees Relief Buying After Extended Drop",
    "subreddit": "RWATimes",
    "score": 1,
    "url": "https://www.reddit.com/r/RWATimes/comments/1pywg3g/indian_phosphate_limited_sees_relief_buying_after/",
    "created_utc": 1767038409.0,
    "author": "rwatimes",
    "num_comments": 0,
    "selftext": "![](https://asserts.btccryptcion.shop/asserts/indian/images/watermark_20250106_33_w1.jpg)\n\nmoradabadvocals> 2025> Indian Phosphate Limited Sees Relief Buying After Extended Drop - Covered Call Writing & Outstanding Profit StrategiesIndian Phosphate Limited Sees Relief Buying After Extended  ...\n\n **Details:**\n- **Published:** 29/12/2025 17:13 (UTC)\n- **\ud83d\udcca Characteristics Score:**\n  > **Asset Type:** *others*\n  > **Sentiment:** `-0.3`\n  > **Entropy:** `0.75`\n  > **Relevance:** `0.85`\n  > **Staleness:** `0.9`\n  > **Uncertainty:** `0.2`\n  > **Level-1 Focus:** *institutional-adoption, blockchain-usage, lega

It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.

It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.

It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.



[
  {
    "id": "1pywkco",
    "title": "[Friedman] Yegor Chinakhov is being dealt from Columbus to Pittsburgh for draft picks",
    "subreddit": "penguins",
    "score": 13,
    "url": "https://i.redd.it/8u9f5bs597ag1.jpeg",
    "created_utc": 1767038660.0,
    "author": "bi_and_busy",
    "num_comments": 5,
    "selftext": ""
  },
  {
    "id": "1pywhky",
    "title": "Marantz hyped book... Turns out a single Keith Gill tweet has more GME impact than moon man's entire cult paperback",
    "subreddit": "gme_meltdown",
    "score": 6,
    "url": "https://i.redd.it/hkhgncyn87ag1.jpeg",
    "created_utc": 1767038493.0,
    "author": "Brick-Lanky",
    "num_comments": 1,
    "selftext": ""
  },
  {
    "id": "1pywfn5",
    "title": "Rachael Lange probably didn\u2019t delete the comment with some of her nasty tweets because some delulu fan defends her. What they said makes sense IF she changed and clearly she hasn\u2019t. She only stopped posting her evil because she got caught, but she st

It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.

It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.



[
  {
    "id": "1pywph6",
    "title": "OZ Deck",
    "subreddit": "GundamTCG",
    "score": 1,
    "url": "https://www.reddit.com/r/GundamTCG/comments/1pywph6/oz_deck/",
    "created_utc": 1767038987.0,
    "author": "spyka78",
    "num_comments": 0,
    "selftext": "Don\u2019t really plan on putting this deck together, but I recently got into the gundam tcg and I was bored at work so I asked ai to throw together a 50 card deck. After a few prompts it came up with this \n\nOZ White Tempo \u2013 50 Cards\n\n\ud83d\udd27 Units (24)\n\nThese win games by curve discipline and pressure.\n\n\t\u2022\t4\u00d7 OZ Leo (Cost 1)\n\nEarly board, pilot host, expendable attacker\n\n\t\u2022\t4\u00d7 OZ Aries (Cost 2)\n\nTempo unit, trades up, pressures life\n\n\t\u2022\t3\u00d7 OZ Tragos (Cost 2\u20133)\n\nDefensive glue, forces awkward attacks\n\n\t\u2022\t3\u00d7 Tallgeese (Cost 3) \u2b50\n\nThe deck\u2019s executioner. Never play 4.\n\n\t\u2022\t2\u00d7 Tallgeese II / variant (Cost 4)\n\nLate-g